# 🧪 Test de l'inférence du modèle LexiorGpt1-merged

Ce notebook vous permet de charger et de tester localement ou sur un serveur de calcul votre modèle fusionné **`intelliwork/LexiorGpt1-merged`**.

### Spécifications matérielles recommandées :
- **Mode 16-bit (sans quantification)** : Nécessite environ ~65 Go de VRAM (ex. A100 80GB, H100).
- **Mode 4-bit (quantifié)** : Nécessite environ ~20 Go de VRAM (ex. RTX 3090/4090, A10G, Mac Studio Apple Silicon).

## 1. Installation des dépendances
Exécutez cette cellule pour vous assurer que les paquets requis sont installés.

In [ ]:
!pip install -q transformers accelerate bitsandbytes torch sentencepiece

## 2. Chargement du modèle et du tokenizer
Configurez la cellule ci-dessous selon votre matériel. Par défaut, elle est configurée en **chargement quantifié 4-bit** (fortement recommandé sur les machines de taille moyenne).

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "intelliwork/LexiorGpt1-merged"

# CONFIGURATION : Choisissez le mode de chargement
LOAD_IN_4BIT = True # Changez à False si vous utilisez un GPU de 80 Go (ex. A100)

print(f"Chargement du tokenizer pour {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

if LOAD_IN_4BIT:
    print("Configuration de la quantification 4-bit (double quantification, nf4)...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True
    )
    print(f"Chargement du modèle quantifié en 4-bit...")
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto"
    )
else:
    print(f"Chargement du modèle en 16-bit complet...")
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="auto"
    )

print("Modèle chargé avec succès !")

## 3. Fonction de génération d'inférence (Chat Template)
Nous utilisons le template standard de Qwen2.5-Instruct pour formater correctement les entrées.

In [ ]:
def generate_response(prompt, system_prompt="You are a helpful assistant.", max_new_tokens=1024, temperature=0.7):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]
    
    # Application du template de chat Qwen
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    # Tokenisation
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    # Génération
    print("Génération de la réponse en cours...")
    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1
        )
    
    # Extraction de la réponse générée
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response

## 4. Test d'inférence interactif
Saisissez votre question juridique ou générale ci-dessous et exécutez la cellule pour voir la réponse du modèle.

In [ ]:
# Exemple de question juridique / CoT
question = "Un employeur au Québec peut-il modifier unilatéralement les heures de travail d'un employé ? Explique ton raisonnement étape par étape."

system_message = "You are LexiorGpt, a specialized legal AI assistant. Always provide structured, step-by-step reasoning (Chain of Thought)."

reponse = generate_response(
    prompt=question,
    system_prompt=system_message,
    max_new_tokens=1500,
    temperature=0.3
)

print("\n=== RÉPONSE DE LEXIORGPT ===\n")
print(reponse)